# LS-SSDD-v1.0 preprocessing for YOLOv8s (Kaggle)

This notebook turns LS-SSDD-v1.0 into a YOLO dataset and shows 4 sample tiles after each stage.

**Stages**
1. Find the dataset and parse the XML annotations
2. Look at the raw tiles with their boxes
3. Normalize brightness (fixed contrast stretch)
4. Speckle filter (shown for comparison, off by default)
5. Convert boxes to YOLO format and check them
6. Split into train / val / test and drop most empty tiles from train
7. Write the dataset and `data.yaml`
8. Preview the training augmentations YOLO will apply

**Output** (in `/kaggle/working` if using Kaggle):
- `lsssdd_yolo/`: the YOLO dataset (images, labels, `data.yaml`, `stretch.txt`)
- `figures/`: a PNG of every stage's figure

In [ ]:
!pip install -q ultralytics

## Settings

In [ ]:
import os, re, random, shutil, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, cv2, matplotlib.pyplot as plt

INPUT_ROOT = Path("/kaggle/input")          # searched for the LS-SSDD files
OUT        = Path("/kaggle/working/lsssdd_yolo")
FIG_DIR    = Path("/kaggle/working/figures")   # every stage's figure is saved here as a PNG

# Splits. LS-SSDD has 15 large scenes, each cut into 600 tiles of 800x800.
# Official split: scenes 1-10 train, 11-15 test. I hold out scene 10 as validation,
# so val never shares a scene with train (neighbouring tiles would leak).
VAL_SCENES  = {10}
TEST_SCENES = {11, 12, 13, 14, 15}

KEEP_EMPTY_FRAC_TRAIN = 0.15   # keep 15% of ship-free tiles in train; val/test keep all
APPLY_STRETCH = True           # fixed contrast stretch (stage 3)
CLIP_PCT      = (0.5, 99.9)    # percentiles used for the stretch
APPLY_SPECKLE = False          # Lee filter (stage 4). Off: it can smear tiny ships.
N_SAMPLES     = 4
SEED          = 0
random.seed(SEED); np.random.seed(SEED)
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_show():
    # Save the current figure to FIG_DIR (named after its title), then display it.
    fig = plt.gcf(); plt.tight_layout()
    title = fig._suptitle.get_text() if fig._suptitle else fig.axes[0].get_title()
    name = re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:60]
    fig.savefig(FIG_DIR / f"{name}.png", dpi=110, bbox_inches="tight")
    plt.show()

## Stage 1: find the files and parse the annotations
LS-SSDD comes with Pascal VOC XML files, one per tile. Each `<object>` holds one ship box as `xmin, ymin, xmax, ymax` in pixels.


In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
TILE_RE = re.compile(r"^(\d+)_(\d+)_(\d+)$")

imgs, xmls = {}, {}
for p in INPUT_ROOT.rglob("*"):
    if not p.is_file() or not TILE_RE.match(p.stem):
        continue
    if p.suffix.lower() in IMG_EXT:
        imgs.setdefault(p.stem, p)
    elif p.suffix.lower() == ".xml":
        xmls.setdefault(p.stem, p)

stems = sorted(set(imgs) & set(xmls))
print(f"tile images: {len(imgs)}   tile xmls: {len(xmls)}   matched pairs: {len(stems)}")
assert stems, "No LS-SSDD tiles found. Check that the dataset is attached under /kaggle/input."
missing = len(imgs) - len(stems)
if missing:
    print(f"warning: {missing} images have no XML and will be skipped")

def parse_voc(path):
    root = ET.parse(path).getroot()
    boxes = []
    for obj in root.iter("object"):
        bb = obj.find("bndbox")
        x1, y1, x2, y2 = (float(bb.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
        boxes.append((x1, y1, x2, y2))
    size = root.find("size")
    wh = (int(size.find("width").text), int(size.find("height").text)) if size is not None else None
    return boxes, wh

ann = {s: parse_voc(xmls[s]) for s in stems}
scene_of = {s: int(TILE_RE.match(s).group(1)) for s in stems}

n_boxes = sum(len(b) for b, _ in ann.values())
n_empty = sum(1 for b, _ in ann.values() if not b)
print(f"ships: {n_boxes}   tiles with ships: {len(stems)-n_empty}   empty tiles: {n_empty}")
print("tiles per scene:", dict(sorted(Counter(scene_of.values()).items())))

### Pick the samples
Three tiles with ships (a few, some, many) and one empty tile, so you see the full range. The same 4 tiles are used in every stage.

In [ ]:
def read_gray(stem):
    im = cv2.imread(str(imgs[stem]), cv2.IMREAD_UNCHANGED)
    if im.ndim == 3:                      # some copies are saved as 3 identical channels
        im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    return im

with_ships = sorted((s for s in stems if ann[s][0]), key=lambda s: len(ann[s][0]))
empty      = [s for s in stems if not ann[s][0]]
picks = [with_ships[len(with_ships)//4], with_ships[len(with_ships)//2], with_ships[-1]]
if empty:
    picks.append(random.choice(empty))
SAMPLES = picks[:N_SAMPLES]
print("samples:", [(s, len(ann[s][0])) for s in SAMPLES])

def show_row(images, titles, suptitle, boxes=None, color=(0, 255, 0), cmap="gray"):
    fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 4.8))
    for i, (ax, im, t) in enumerate(zip(np.atleast_1d(axes), images, titles)):
        if boxes is not None:
            im = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB) if im.ndim == 2 else im.copy()
            for (x1, y1, x2, y2) in boxes[i]:
                cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            ax.imshow(im)
        else:
            ax.imshow(im, cmap=cmap, vmin=0, vmax=255)
        ax.set_title(t, fontsize=10); ax.axis("off")
    fig.suptitle(suptitle, fontsize=13); save_show()

## Stage 2: raw tiles and their boxes
This is the data as it comes. Things to notice:
- The sea is dark and grainy. That grain is **speckle**, a noise every SAR image has.
- Ships are small bright dots, often only 5 to 20 pixels long. That's what 10 m pixels do to a 100 m ship.
- The green boxes come straight from the XML files.

In [ ]:
raw = [read_gray(s) for s in SAMPLES]
show_row(raw, [f"{s}  ({len(ann[s][0])} ships)" for s in SAMPLES], "Stage 2a: raw tiles")
show_row(raw, [f"{s}  ({len(ann[s][0])} ships)" for s in SAMPLES], "Stage 2b: raw tiles + XML boxes",
         boxes=[ann[s][0] for s in SAMPLES])

## Stage 3: brightness normalization
Most pixels are dark sea, so the image uses only a small part of the 0–255 range. A **contrast stretch** fixes that: pick a low and a high brightness value, map them to 0 and 255, and clip everything outside.

The low/high values are computed **once from the training tiles** and then used for every tile. A per-tile stretch would turn empty, calm sea into bright noise, and it would make the same ship look different from tile to tile.

**Important:** use the same two numbers when for preprocessing the Sentinel-1 scene for inference, saved to `stretch.txt`.

In [ ]:
train_stems = [s for s in stems if scene_of[s] not in VAL_SCENES | TEST_SCENES]
probe = random.sample(train_stems, min(300, len(train_stems)))
pix = np.concatenate([read_gray(s)[::4, ::4].ravel() for s in probe])
LO, HI = np.percentile(pix, CLIP_PCT)
if HI <= LO: HI = LO + 1
print(f"stretch: {CLIP_PCT[0]}th pct = {LO:.1f}, {CLIP_PCT[1]}th pct = {HI:.1f}  (from {len(probe)} train tiles)")

def stretch(im):
    return np.clip((im.astype(np.float32) - LO) * 255.0 / (HI - LO), 0, 255).astype(np.uint8)

stretched = [stretch(im) for im in raw]
show_row(stretched, SAMPLES, "Stage 3a: after fixed contrast stretch")

fig, axes = plt.subplots(1, len(SAMPLES), figsize=(4.5 * len(SAMPLES), 3))
for ax, a, b, s in zip(axes, raw, stretched, SAMPLES):
    ax.hist(a.ravel(), bins=64, range=(0, 255), alpha=.6, label="raw", log=True)
    ax.hist(b.ravel(), bins=64, range=(0, 255), alpha=.6, label="stretched", log=True)
    ax.set_title(s, fontsize=10); ax.legend(fontsize=8)
fig.suptitle("Stage 3b: pixel histograms (log scale)"); save_show()

## Stage 4: speckle filter (for comparison only)
A **Lee filter** smooths speckle in flat areas (sea) and leaves edges alone. It makes the image look cleaner, but the zoomed crops below show the catch: tiny ships get softer and dimmer too. That's why it's off by default. YOLO learns to cope with speckle on its own.

In [ ]:
def lee_filter(im, k=5):
    img = im.astype(np.float32)
    mean = cv2.blur(img, (k, k))
    var = np.maximum(cv2.blur(img * img, (k, k)) - mean * mean, 0)
    noise_var = np.mean(var)
    w = var / (var + noise_var + 1e-6)
    return np.clip(mean + w * (img - mean), 0, 255).astype(np.uint8)

filtered = [lee_filter(im) for im in stretched]
show_row(filtered, SAMPLES, "Stage 4a: after Lee filter (5x5)")

def crop_around(im, box, half=48):
    cx, cy = int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2)
    h, w = im.shape[:2]
    x0, y0 = min(max(cx - half, 0), w - 2*half), min(max(cy - half, 0), h - 2*half)
    return cv2.resize(im[y0:y0+2*half, x0:x0+2*half], (288, 288), interpolation=cv2.INTER_NEAREST)

ship_idx = [i for i, s in enumerate(SAMPLES) if ann[s][0]]
fig, axes = plt.subplots(2, len(ship_idx), figsize=(4 * len(ship_idx), 8))
for j, i in enumerate(ship_idx):
    box = ann[SAMPLES[i]][0][0]
    axes[0, j].imshow(crop_around(stretched[i], box), cmap="gray", vmin=0, vmax=255); axes[0, j].set_title("no filter, zoomed")
    axes[1, j].imshow(crop_around(filtered[i], box), cmap="gray", vmin=0, vmax=255);  axes[1, j].set_title("Lee filter, zoomed")
    for a in axes[:, j]: a.axis("off")
fig.suptitle("Stage 4b: 96x96 crop around one ship, enlarged 3x"); save_show()

def preprocess(im):
    if APPLY_STRETCH: im = stretch(im)
    if APPLY_SPECKLE: im = lee_filter(im)
    return im

## Stage 5: convert boxes to YOLO format
YOLO wants one `.txt` file per image, one line per ship:

```
class  x_center  y_center  width  height      (all divided by the image size, so 0 to 1)
```

Example: a box `xmin=400, ymin=300, xmax=420, ymax=310` on an 800×800 tile becomes `0 0.5125 0.38125 0.025 0.0125`. Empty tiles get an empty file. That tells YOLO "no ships here", which is useful.

To check the conversion, the boxes below are drawn **back from the YOLO numbers** (red). They should sit exactly where the green XML boxes were.

In [ ]:
def to_yolo(boxes, w, h):
    lines = []
    for x1, y1, x2, y2 in boxes:
        x1, x2 = max(0, min(x1, w)), max(0, min(x2, w))
        y1, y2 = max(0, min(y1, h)), max(0, min(y2, h))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue                           # drop broken/degenerate boxes
        lines.append(f"0 {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
    return lines

def from_yolo(lines, w, h):
    out = []
    for ln in lines:
        _, cx, cy, bw, bh = map(float, ln.split())
        out.append(((cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h))
    return out

sample_pre = [preprocess(im) for im in raw]
yolo_lines = [to_yolo(ann[s][0], im.shape[1], im.shape[0]) for s, im in zip(SAMPLES, raw)]
for s, l in zip(SAMPLES, yolo_lines):
    print(f"{s}.txt  ({len(l)} lines)", *l[:3], "..." if len(l) > 3 else "", sep="\n  ")
show_row(sample_pre, SAMPLES, "Stage 5a: preprocessed tile + boxes rebuilt from YOLO labels",
         boxes=[from_yolo(l, im.shape[1], im.shape[0]) for l, im in zip(yolo_lines, raw)], color=(255, 0, 0))

### How big are the ships?
This is why we train at `imgsz=800` and don't shrink the tiles. Anything under 32 px counts as a "small object" in COCO terms.

In [ ]:
ws = np.array([b[2]-b[0] for s in stems for b in ann[s][0]])
hs = np.array([b[3]-b[1] for s in stems for b in ann[s][0]])
side = np.sqrt(ws * hs)
print(f"box size (sqrt of area): median {np.median(side):.1f} px, "
      f"{(side < 16).mean()*100:.0f}% under 16 px, {(side < 32).mean()*100:.0f}% under 32 px")
plt.figure(figsize=(7, 3.5))
plt.hist(side, bins=np.arange(0, max(64, side.max()+2), 2))
plt.axvline(32, color="r", ls="--", label="32 px (COCO 'small')")
plt.xlabel("box size, sqrt(w*h) in pixels"); plt.ylabel("ships"); plt.legend()
plt.title("Stage 5b: ship size distribution"); save_show()

## Stage 6: split and filter empty tiles
- **train:** scenes 1–9
- **val:** scene 10 (used during training to pick the best epoch)
- **test:** scenes 11–15, the official LS-SSDD test set. Report your final numbers on this.

Most tiles are open sea with no ships. Training on all of them wastes time and teaches the model that "nothing" is almost always the answer. So train keeps only 15% of the empty tiles. Val and test keep everything, so the scores stay honest.

In [ ]:
def split_of(s):
    sc = scene_of[s]
    return "test" if sc in TEST_SCENES else "val" if sc in VAL_SCENES else "train"

splits = {"train": [], "val": [], "test": []}
for s in stems:
    splits[split_of(s)].append(s)

before = {k: (sum(1 for s in v if ann[s][0]), sum(1 for s in v if not ann[s][0])) for k, v in splits.items()}
tr_empty = [s for s in splits["train"] if not ann[s][0]]
keep_empty = set(random.sample(tr_empty, int(len(tr_empty) * KEEP_EMPTY_FRAC_TRAIN)))
splits["train"] = [s for s in splits["train"] if ann[s][0] or s in keep_empty]
after = {k: (sum(1 for s in v if ann[s][0]), sum(1 for s in v if not ann[s][0])) for k, v in splits.items()}

for k in splits:
    print(f"{k:5s}: {before[k][0]:5d} with ships + {before[k][1]:5d} empty  ->  "
          f"{after[k][0]:5d} with ships + {after[k][1]:5d} empty   ({sum(len(ann[s][0]) for s in splits[k])} ships)")

fig, ax = plt.subplots(figsize=(7, 3.5)); x = np.arange(3); names = list(splits)
ax.bar(x - .2, [after[k][0] for k in names], .4, label="tiles with ships")
ax.bar(x + .2, [after[k][1] for k in names], .4, label="empty tiles (after filtering)")
ax.set_xticks(x, names); ax.legend(); ax.set_title("Stage 6: tiles per split"); save_show()

## Stage 7: write the YOLO dataset
Layout:
```
lsssdd_yolo/
  images/{train,val,test}/*.png   preprocessed tiles (PNG, so no extra JPEG blur)
  labels/{train,val,test}/*.txt   YOLO boxes
  data.yaml
  stretch.txt                     LO/HI values to reuse at inference
```

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
for k in splits:
    (OUT / "images" / k).mkdir(parents=True); (OUT / "labels" / k).mkdir(parents=True)

def write_one(args):
    k, s = args
    im = read_gray(s)
    h, w = im.shape[:2]
    cv2.imwrite(str(OUT / "images" / k / f"{s}.png"), preprocess(im))
    (OUT / "labels" / k / f"{s}.txt").write_text("\n".join(to_yolo(ann[s][0], w, h)))

jobs = [(k, s) for k, v in splits.items() for s in v]
with ThreadPoolExecutor(8) as ex:
    list(ex.map(write_one, jobs))

(OUT / "data.yaml").write_text(
    f"path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnc: 1\nnames: ['ship']\n")
(OUT / "stretch.txt").write_text(
    f"APPLY_STRETCH={APPLY_STRETCH}\nLO={LO}\nHI={HI}\nAPPLY_SPECKLE={APPLY_SPECKLE}\n")
for k in splits:
    print(k, len(list((OUT / "images" / k).iterdir())), "images,", len(list((OUT / "labels" / k).iterdir())), "labels")
print((OUT / "data.yaml").read_text())

## Stage 8: preview the training augmentations
YOLO does augmentation itself, on the fly, during training. These settings suit SAR:
- **flips** up/down and left/right: a ship seen from above looks valid either way.
- **mosaic**: stitches 4 tiles into one, so the model sees more ships per step.
- **scale ±50%**: ships of different sizes.
- **no hue/saturation changes**: SAR has no colour, so those do nothing useful. Brightness gets a small change only.

In [ ]:
AUG = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, degrees=0.0, translate=0.1, scale=0.5,
           fliplr=0.5, flipud=0.5, mosaic=1.0, mixup=0.0)

from ultralytics.cfg import get_cfg
from ultralytics.data.build import build_yolo_dataset
from ultralytics.data.utils import check_det_dataset

cfg = get_cfg(overrides=dict(imgsz=800, **AUG))
ds = build_yolo_dataset(cfg, str(OUT / "images" / "train"), batch=4,
                        data=check_det_dataset(str(OUT / "data.yaml")), mode="train")
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, i in zip(axes, random.sample(range(len(ds)), 4)):
    d = ds[i]
    im = d["img"].numpy().transpose(1, 2, 0)[:, :, ::-1].copy()
    H, W = im.shape[:2]
    for cx, cy, bw, bh in d["bboxes"].numpy():
        cv2.rectangle(im, (int((cx-bw/2)*W), int((cy-bh/2)*H)), (int((cx+bw/2)*W), int((cy+bh/2)*H)), (255, 0, 0), 2)
    ax.imshow(im); ax.set_title(f"augmented sample, {len(d['bboxes'])} ships", fontsize=10); ax.axis("off")
fig.suptitle("Stage 8: what YOLO sees during training (mosaic + flips + scale)"); save_show()